In [0]:
%run ../configs/log_configs

In [0]:
# ─────────────────────────────────────────────
# Cell 2b: Install Gemini SDK
# Run once — cached after first cluster start
# ─────────────────────────────────────────────
%pip install google-generativeai --quiet

In [0]:
# ─────────────────────────────────────────────
# Cell 2c: LLM Connection — Cached
# ─────────────────────────────────────────────
import google.generativeai as genai

# ── Configure once ─────────────────────────
genai.configure(api_key=gemini_api_key)

# ── Model name from Key Vault ──────────────
# No hardcoding — change model in vault only
_LLM_MODEL = genai.GenerativeModel(
    model_name=gemini_model_name,
    generation_config={
        "temperature":       0.2,
        "max_output_tokens": 500,
        "top_p":             0.8
    }
)

print(f"LLM model cached    : {gemini_model_name}")
print(f"API configured      : Gemini")


def get_llm_insight(prompt: str) -> str:
    try:
        response = _LLM_MODEL.generate_content(prompt)
        return response.text.strip()
    except Exception as llm_err:
        print(f"LLM call failed: {str(llm_err)}")
        return f"Insight unavailable: {str(llm_err)}"

In [0]:
def generate_product_insight(df):
    """
    Analyzes product regional performance
    and generates product strategy insight
    """
    stats = df.agg(
        F.sum("total_holdings").alias("total_holdings"),
        F.sum("total_value").alias("total_value"),
        F.avg("active_rate").alias("avg_active_rate"),
        F.avg("campaign_conversion_rate")
         .alias("avg_conv_rate")
    ).collect()[0]

    # ── Top product by total value ──────────
    top_product = (
        df.groupBy("product_name")
        .agg(F.sum("total_value").alias("val"))
        .orderBy(F.desc("val"))
        .first()["product_name"]
    )

    # ── Top zone by holdings ────────────────
    top_zone = (
        df.groupBy("zone")
        .agg(F.sum("total_holdings").alias("cnt"))
        .orderBy(F.desc("cnt"))
        .first()["zone"]
    )

    # ── Lowest performing product ───────────
    low_product = (
        df.groupBy("product_name")
        .agg(F.avg("active_rate").alias("rate"))
        .orderBy(F.asc("rate"))
        .first()["product_name"]
    )

    prompt = f"""
You are a banking product strategy analyst.
Analyze this Product Regional Performance summary
for Bharat Indus Bank and give a 4-5 line insight:

- Total Holdings Across All Products : {stats['total_holdings']}
- Total Portfolio Value              : ₹{round(stats['total_value'] or 0, 2)}
- Average Active Rate                : {round(stats['avg_active_rate'] or 0, 2)}%
- Avg Campaign Conversion Rate       : {round(stats['avg_conv_rate'] or 0, 2)}%
- Top Product by Value               : {top_product}
- Top Zone by Holdings               : {top_zone}
- Lowest Active Rate Product         : {low_product}

Focus on: which products are gaining traction,
geographic concentration risk, and one
cross-sell recommendation for the bank.
    """
    return get_llm_insight(prompt)

In [0]:
# ─────────────────────────────────────────────
# Cell 1: Imports
# ─────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

today = datetime.now()
year  = today.strftime("%Y")
month = today.strftime("%m")
day   = today.strftime("%d")

In [0]:
# ─────────────────────────────────────────────
# Cell 2: Widgets
# ─────────────────────────────────────────────
dbutils.widgets.text("environment",   "dev")
dbutils.widgets.text("layer",         "gold")
dbutils.widgets.text("notebook_name", "03_gold_aggregation_runner")
dbutils.widgets.text("pipeline_name", "BIB_Data_processing")

environment   = dbutils.widgets.get("environment")
layer         = dbutils.widgets.get("layer")
notebook_name = dbutils.widgets.get("notebook_name")
pipeline_name = dbutils.widgets.get("pipeline_name")

print(f"Environment : {environment}")
print(f"Layer       : {layer}")
print(f"Pipeline    : {pipeline_name}")

In [0]:
# ─────────────────────────────────────────────
# Cell 2d: AI Insight Generators
# Called after each gold table is built
# Results written to bib_catalog.gold.ai_insights
# ─────────────────────────────────────────────

def generate_customer_360_insight(df):
    """
    Analyzes customer 360 gold table
    and generates executive summary
    """
    # ── Compute key stats to feed LLM ──────
    stats = df.agg(
        F.count("customer_id")
         .alias("total_customers"),
        F.avg("total_txn_amount")
         .alias("avg_txn_amount"),
        F.avg("engagement_score")
         .alias("avg_engagement"),
        F.avg("total_holdings")
         .alias("avg_holdings"),
        F.sum(
            F.when(
                F.col("kyc_status") == "Verified", 1
            ).otherwise(0)
        ).alias("verified_customers")
    ).collect()[0]

    # ── Top segment ────────────────────────
    top_segment = (
        df.groupBy("segment")
        .count()
        .orderBy(F.desc("count"))
        .first()["segment"]
    )

    # ── Top zone ───────────────────────────
    top_zone = (
        df.groupBy("zone")
        .agg(F.sum("total_txn_amount").alias("val"))
        .orderBy(F.desc("val"))
        .first()["zone"]
    )

    prompt = f"""
You are a banking data analyst.
Analyze this Customer 360 summary for Bharat Indus Bank
and give a concise 4-5 line executive insight:

- Total Customers       : {stats['total_customers']}
- Avg Transaction Amount: ₹{round(stats['avg_txn_amount'] or 0, 2)}
- Avg Engagement Score  : {round(stats['avg_engagement'] or 0, 2)}
- Avg Product Holdings  : {round(stats['avg_holdings'] or 0, 2)}
- KYC Verified          : {stats['verified_customers']}
- Top Customer Segment  : {top_segment}
- Top Zone by Revenue   : {top_zone}

Focus on: customer health, engagement trends,
and one actionable recommendation for the bank.
Keep it professional and data-driven.
    """
    return get_llm_insight(prompt)


def generate_transaction_insight(df):
    """
    Analyzes transaction intelligence gold table
    """
    stats = df.agg(
        F.sum("txn_count").alias("total_txns"),
        F.sum("total_amount").alias("total_amount"),
        F.avg("avg_amount").alias("avg_amount")
    ).collect()[0]

    top_type = (
        df.groupBy("txn_type")
        .agg(F.sum("txn_count").alias("cnt"))
        .orderBy(F.desc("cnt"))
        .first()["txn_type"]
    )

    top_channel = (
        df.groupBy("channel")
        .agg(F.sum("txn_count").alias("cnt"))
        .orderBy(F.desc("cnt"))
        .first()["channel"]
    )

    top_merchant = (
        df.groupBy("merchant_category")
        .agg(F.sum("total_amount").alias("val"))
        .orderBy(F.desc("val"))
        .first()["merchant_category"]
    )

    prompt = f"""
You are a banking data analyst.
Analyze this Transaction Intelligence summary
for Bharat Indus Bank and give a 4-5 line insight:

- Total Transactions    : {stats['total_txns']}
- Total Amount          : ₹{round(stats['total_amount'] or 0, 2)}
- Average Amount        : ₹{round(stats['avg_amount'] or 0, 2)}
- Top Transaction Type  : {top_type}
- Top Channel           : {top_channel}
- Top Merchant Category : {top_merchant}

Focus on: spending patterns, channel preference,
and one risk or opportunity observation.
    """
    return get_llm_insight(prompt)


def generate_campaign_insight(df):
    """
    Analyzes campaign performance gold table
    """
    stats = df.agg(
        F.sum("sent_count").alias("total_sent"),
        F.sum("converted_count").alias("total_conv"),
        F.avg("conversion_rate").alias("avg_conv_rate"),
        F.avg("open_rate").alias("avg_open_rate")
    ).collect()[0]

    top_campaign = (
        df.groupBy("campaign_name")
        .agg(F.sum("converted_count").alias("conv"))
        .orderBy(F.desc("conv"))
        .first()["campaign_name"]
    )

    worst_campaign = (
        df.groupBy("campaign_name")
        .agg(F.avg("conversion_rate").alias("rate"))
        .orderBy(F.asc("rate"))
        .first()["campaign_name"]
    )

    prompt = f"""
You are a banking marketing analyst.
Analyze this Campaign Performance summary
for Bharat Indus Bank and give a 4-5 line insight:

- Total Campaigns Sent      : {stats['total_sent']}
- Total Conversions         : {stats['total_conv']}
- Average Conversion Rate   : {round(stats['avg_conv_rate'] or 0, 2)}%
- Average Open Rate         : {round(stats['avg_open_rate'] or 0, 2)}%
- Best Performing Campaign  : {top_campaign}
- Needs Improvement         : {worst_campaign}

Focus on: campaign effectiveness, conversion funnel,
and one actionable improvement recommendation.
    """
    return get_llm_insight(prompt)


def generate_branch_insight(df):
    """
    Analyzes branch performance gold table
    """
    stats = df.agg(
        F.sum("total_customers").alias("customers"),
        F.sum("total_txn_value").alias("txn_value"),
        F.avg("total_portfolio_value").alias("avg_portfolio")
    ).collect()[0]

    top_branch = (
        df.orderBy(F.desc("total_txn_value"))
        .first()["branch_name"]
    )

    top_zone = (
        df.groupBy("zone")
        .agg(F.sum("total_txn_value").alias("val"))
        .orderBy(F.desc("val"))
        .first()["zone"]
    )

    prompt = f"""
You are a banking operations analyst.
Analyze this Branch Performance summary
for Bharat Indus Bank and give a 4-5 line insight:

- Total Customers (all branches) : {stats['customers']}
- Total Transaction Value        : ₹{round(stats['txn_value'] or 0, 2)}
- Avg Portfolio Value per Branch : ₹{round(stats['avg_portfolio'] or 0, 2)}
- Top Performing Branch          : {top_branch}
- Top Performing Zone            : {top_zone}

Focus on: branch productivity, geographic strengths,
and one recommendation for underperforming branches.
    """
    return get_llm_insight(prompt)


def generate_anomaly_detection_insight(
    transactions_df, threshold_multiplier=2.0
):
    """
    AI-powered anomaly detection on transactions.
    Flags statistically unusual transaction amounts.
    This is above and beyond — AI inside the pipeline.
    """
    # ── Compute stats ──────────────────────
    stats = transactions_df.agg(
        F.avg("total_amount").alias("mean"),
        F.stddev("total_amount").alias("stddev"),
        F.max("total_amount").alias("max_val"),
        F.min("total_amount").alias("min_val"),
        F.count("*").alias("total_records")
    ).collect()[0]

    mean   = stats["mean"]   or 0
    stddev = stats["stddev"] or 0
    threshold = mean + (threshold_multiplier * stddev)

    # ── Count anomalies ────────────────────
    anomaly_count = transactions_df.filter(
        F.col("total_amount") > threshold
    ).count()

    # ── Top anomalous categories ───────────
    top_anomaly_category = (
        transactions_df
        .filter(F.col("total_amount") > threshold)
        .groupBy("merchant_category")
        .count()
        .orderBy(F.desc("count"))
        .first()
    )
    top_cat = (
        top_anomaly_category["merchant_category"]
        if top_anomaly_category else "N/A"
    )

    prompt = f"""
You are a banking fraud and risk analyst.
Analyze these transaction anomaly statistics
for Bharat Indus Bank:

- Total Records Analyzed    : {stats['total_records']}
- Mean Transaction Amount   : ₹{round(mean, 2)}
- Std Deviation             : ₹{round(stddev, 2)}
- Anomaly Threshold (2σ)    : ₹{round(threshold, 2)}
- Anomalous Transactions    : {anomaly_count}
- Top Anomaly Category      : {top_cat}
- Max Transaction Amount    : ₹{round(stats['max_val'] or 0, 2)}

Provide a 4-5 line risk assessment:
- Are these anomalies concerning?
- What could explain the pattern?
- What action should the risk team take?
    """
    return get_llm_insight(prompt)

In [0]:
def generate_product_insight(df):
    """
    Analyzes product regional performance
    and generates product strategy insight
    """
    stats = df.agg(
        F.sum("total_holdings").alias("total_holdings"),
        F.sum("total_value").alias("total_value"),
        F.avg("active_rate").alias("avg_active_rate"),
        F.avg("campaign_conversion_rate")
         .alias("avg_conv_rate")
    ).collect()[0]

    # ── Top product by total value ──────────
    top_product = (
        df.groupBy("product_name")
        .agg(F.sum("total_value").alias("val"))
        .orderBy(F.desc("val"))
        .first()["product_name"]
    )

    # ── Top zone by holdings ────────────────
    top_zone = (
        df.groupBy("zone")
        .agg(F.sum("total_holdings").alias("cnt"))
        .orderBy(F.desc("cnt"))
        .first()["zone"]
    )

    # ── Lowest performing product ───────────
    low_product = (
        df.groupBy("product_name")
        .agg(F.avg("active_rate").alias("rate"))
        .orderBy(F.asc("rate"))
        .first()["product_name"]
    )

    prompt = f"""
You are a banking product strategy analyst.
Analyze this Product Regional Performance summary
for Bharat Indus Bank and give a 4-5 line insight:

- Total Holdings Across All Products : {stats['total_holdings']}
- Total Portfolio Value              : ₹{round(stats['total_value'] or 0, 2)}
- Average Active Rate                : {round(stats['avg_active_rate'] or 0, 2)}%
- Avg Campaign Conversion Rate       : {round(stats['avg_conv_rate'] or 0, 2)}%
- Top Product by Value               : {top_product}
- Top Zone by Holdings               : {top_zone}
- Lowest Active Rate Product         : {low_product}

Focus on: which products are gaining traction,
geographic concentration risk, and one
cross-sell recommendation for the bank.
    """
    return get_llm_insight(prompt)

In [0]:
# ─────────────────────────────────────────────
# Cell 2e: AI Insights Table Writer
# Writes all LLM insights to a Gold Delta table
# Queryable via SQL — shown in Power BI + Streamlit
# ─────────────────────────────────────────────
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType
)

def write_ai_insight(
    table_name, insight_text,
    insight_type, source_table
):
    """
    Persists LLM insight to
    bib_catalog.gold.ai_insights Delta table.
    Appends — full history of daily insights kept.
    """
    schema = StructType([
        StructField("insight_id",    StringType(),   True),
        StructField("table_name",    StringType(),   True),
        StructField("insight_type",  StringType(),   True),
        StructField("source_table",  StringType(),   True),
        StructField("insight_text",  StringType(),   True),
        StructField("environment",   StringType(),   True),
        StructField("generated_at",  TimestampType(),True)
    ])

    import uuid
    from datetime import datetime

    row = [(
        str(uuid.uuid4()),
        table_name,
        insight_type,
        source_table,
        insight_text,
        environment,
        datetime.now()
    )]

    insight_df = spark.createDataFrame(row, schema)

    # ── Ensure schema exists ───────────────
    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS "
        "bib_catalog.gold"
    )

    # ── Append to insights table ───────────
    (
        insight_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("bib_catalog.gold.ai_insights")
    )

    # ── Also write to ADLS ─────────────────
    adls_insights_path = get_adls_path(
        environment, layer, "ai_insights/"
    )
    (
        insight_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .save(adls_insights_path)
    )

    print(
        f"AI insight written  : "
        f"{insight_type} for {table_name}"
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 3: Gold Writer Helper
# Writes to Unity Catalog + ADLS
# Always overwrites — Gold is fully recomputed
# ─────────────────────────────────────────────
def write_gold(df, gold_table, z_order_col=None):

    parts  = gold_table.split(".")
    cat    = parts[0]
    schema = parts[1]
    table  = parts[2]

    # ── Ensure schema exists ───────────────
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {cat}.{schema}"
    )

    adls_gold_path = get_adls_path(
        environment, layer, f"{table}/"
    )

    # ── Add gold audit columns ─────────────
    df = (
        df
        .withColumn(
            "gold_refreshed_at",
            F.current_timestamp()
        )
        .withColumn(
            "gold_environment",
            F.lit(environment)
        )
    )

    # ── Write to Unity Catalog ─────────────
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(gold_table)
    )
    print(f"UC write done: {gold_table}")

    # ── Write to ADLS ──────────────────────
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(adls_gold_path)
    )
    print(f"ADLS write done: {adls_gold_path}")

    # ── OPTIMIZE ───────────────────────────
    if z_order_col:
        spark.sql(
            f"OPTIMIZE {gold_table} "
            f"ZORDER BY ({z_order_col})"
        )
    else:
        spark.sql(f"OPTIMIZE {gold_table}")

    spark.sql(
        f"VACUUM {gold_table} RETAIN 168 HOURS"
    )
    print(f"OPTIMIZE done: {gold_table}")

    return df.count()

In [0]:
# ─────────────────────────────────────────────
# Cell 3b: Read Master Data from Raw
# For static reference tables not yet in silver
# ─────────────────────────────────────────────
def read_products_from_raw():
    raw_products_path = get_adls_path(
        environment, "raw",
        "master-data/products/"
    )
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(raw_products_path)
    )

In [0]:
def build_customer_360():

    print("Building Customer 360...")

    # ── Read Silver tables ─────────────────
    customers = spark.read.table(
        "bib_catalog.silver.customers"
    ).filter(F.col("scd_is_current") == True)

    accounts = spark.read.table(
        "bib_catalog.silver.accounts"
    ).filter(F.col("scd_is_current") == True)

    product_holdings = spark.read.table(
        "bib_catalog.silver.product_holdings"
    ).filter(F.col("scd_is_current") == True)

    transactions = spark.read.table(
        "bib_catalog.silver.transactions"
    )

    digital_activity = spark.read.table(
        "bib_catalog.silver.digital_activity"
    )

    campaign_interactions = spark.read.table(
        "bib_catalog.silver.campaign_interactions"
    )

    branches = spark.read.table(
        "bib_catalog.silver.branches"
    )

    bank_managers = spark.read.table(
        "bib_catalog.silver.bank_managers"
    )

    # ── Transaction summary per customer ───
    txn_summary = (
        transactions
        .groupBy("customer_id")
        .agg(
            F.count("txn_id")
             .alias("total_transactions"),
            F.sum("amount")
             .alias("total_txn_amount"),
            F.avg("amount")
             .alias("avg_txn_amount"),
            F.max("amount")
             .alias("max_txn_amount"),
            F.min("txn_datetime")
             .alias("first_txn_date"),
            F.max("txn_datetime")
             .alias("last_txn_date"),
            F.sum(
                F.when(
                    F.col("status") == "Success", 1
                ).otherwise(0)
            ).alias("successful_txns"),
            F.sum(
                F.when(
                    F.col("status") == "Failed", 1
                ).otherwise(0)
            ).alias("failed_txns"),
            F.sum(
                F.when(
                    F.col("txn_type") == "UPI", 1
                ).otherwise(0)
            ).alias("upi_txns"),
            F.sum(
                F.when(
                    F.col("txn_type") == "NEFT", 1
                ).otherwise(0)
            ).alias("neft_txns"),
            F.sum(
                F.when(
                    F.col("txn_type") == "Card", 1
                ).otherwise(0)
            ).alias("card_txns")
        )
    )

    # ── Product holding summary ────────────
    holdings_summary = (
        product_holdings
        .groupBy("customer_id")
        .agg(
            F.count("holding_id")
             .alias("total_holdings"),
            F.sum("value_amount")
             .alias("total_portfolio_value"),
            F.sum(
                F.when(
                    F.col("status") == "Active", 1
                ).otherwise(0)
            ).alias("active_holdings")
        )
    )

    # ── Digital activity summary ───────────
    digital_summary = (
        digital_activity
        .groupBy("customer_id")
        .agg(
            F.count("activity_id")
             .alias("total_digital_events"),
            F.countDistinct("session_id")
             .alias("total_sessions"),
            F.sum(
                F.when(
                    F.col("channel") == "Mobile Banking", 1
                ).otherwise(0)
            ).alias("mobile_sessions"),
            F.sum(
                F.when(
                    F.col("channel") == "Internet Banking", 1
                ).otherwise(0)
            ).alias("internet_sessions"),
            F.max("event_datetime")
             .alias("last_digital_activity")
        )
    )

    # ── Campaign summary ───────────────────
    campaign_summary = (
        campaign_interactions
        .groupBy("customer_id")
        .agg(
            F.count("interaction_id")
             .alias("total_campaign_interactions"),
            F.sum("converted")
             .alias("total_conversions"),
            F.countDistinct("campaign_id")
             .alias("campaigns_engaged")
        )
    )

    # ── Account summary ────────────────────
    accounts_summary = (
        accounts
        .groupBy("customer_id")
        .agg(
            F.count("holding_id")
             .alias("total_accounts"),
            F.sum("value_amount")
             .alias("total_account_value"),
            F.sum(
                F.when(
                    F.col("status") == "Active", 1
                ).otherwise(0)
            ).alias("active_accounts")
        )
    )

    # ── Join all together ──────────────────
    customer_360 = (                              # ← indented inside function
        customers
        .join(
            branches.select(
                "branch_id", "branch_name",
                "branch_type",
                F.col("zone").alias("branch_zone"),
                F.col("region").alias("branch_region"),
                "city", "state"
            ),
            on="branch_id",
            how="left"
        )
        .join(
            bank_managers.select(
                F.col("manager_id")
                 .alias("mgr_manager_id"),
                F.col("full_name")
                 .alias("manager_name"),
                F.col("role")
                 .alias("manager_role")
            ),
            customers.manager_id == F.col("mgr_manager_id"),
            how="left"
        )
        .join(txn_summary,      on="customer_id", how="left")
        .join(holdings_summary, on="customer_id", how="left")
        .join(digital_summary,  on="customer_id", how="left")
        .join(campaign_summary, on="customer_id", how="left")
        .join(accounts_summary, on="customer_id", how="left")
        .select(
            customers.customer_id,
            customers.full_name,
            customers.gender,
            customers.city,
            customers.state,
            customers.occupation,
            customers.income_band,
            customers.segment,
            customers.kyc_status,
            customers.onboarded_date,
            F.col("branch_name"),
            F.col("branch_type"),
            F.col("branch_zone").alias("zone"),
            F.col("branch_region").alias("region"),
            F.col("manager_name"),
            F.col("manager_role"),
            F.coalesce(F.col("total_transactions"), F.lit(0))
             .alias("total_transactions"),
            F.coalesce(F.col("total_txn_amount"), F.lit(0.0))
             .alias("total_txn_amount"),
            F.coalesce(F.col("avg_txn_amount"), F.lit(0.0))
             .alias("avg_txn_amount"),
            F.coalesce(F.col("max_txn_amount"), F.lit(0.0))
             .alias("max_txn_amount"),
            F.col("first_txn_date"),
            F.col("last_txn_date"),
            F.coalesce(F.col("successful_txns"), F.lit(0))
             .alias("successful_txns"),
            F.coalesce(F.col("failed_txns"), F.lit(0))
             .alias("failed_txns"),
            F.coalesce(F.col("upi_txns"), F.lit(0))
             .alias("upi_txns"),
            F.coalesce(F.col("neft_txns"), F.lit(0))
             .alias("neft_txns"),
            F.coalesce(F.col("card_txns"), F.lit(0))
             .alias("card_txns"),
            F.coalesce(F.col("total_holdings"), F.lit(0))
             .alias("total_holdings"),
            F.coalesce(F.col("total_portfolio_value"), F.lit(0.0))
             .alias("total_portfolio_value"),
            F.coalesce(F.col("active_holdings"), F.lit(0))
             .alias("active_holdings"),
            F.coalesce(F.col("total_accounts"), F.lit(0))
             .alias("total_accounts"),
            F.coalesce(F.col("total_account_value"), F.lit(0.0))
             .alias("total_account_value"),
            F.coalesce(F.col("active_accounts"), F.lit(0))
             .alias("active_accounts"),
            F.coalesce(F.col("total_digital_events"), F.lit(0))
             .alias("total_digital_events"),
            F.coalesce(F.col("total_sessions"), F.lit(0))
             .alias("total_sessions"),
            F.coalesce(F.col("mobile_sessions"), F.lit(0))
             .alias("mobile_sessions"),
            F.coalesce(F.col("internet_sessions"), F.lit(0))
             .alias("internet_sessions"),
            F.col("last_digital_activity"),
            F.coalesce(
                F.col("total_campaign_interactions"),
                F.lit(0)
            ).alias("total_campaign_interactions"),
            F.coalesce(F.col("total_conversions"), F.lit(0))
             .alias("total_conversions"),
            F.coalesce(F.col("campaigns_engaged"), F.lit(0))
             .alias("campaigns_engaged"),
            (
                F.coalesce(
                    F.col("total_digital_events"),
                    F.lit(0)
                ) * 0.4 +
                F.coalesce(
                    F.col("total_transactions"),
                    F.lit(0)
                ) * 0.4 +
                F.coalesce(
                    F.col("total_conversions"),
                    F.lit(0)
                ) * 0.2
            ).alias("engagement_score")
        )
    )

    return write_gold(                            # ← indented inside function
        customer_360,
        "bib_catalog.gold.customer_360",
        z_order_col="customer_id"
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 5: Gold Table 2 — Transaction Intelligence
# Daily + monthly aggregations for Power BI
# ─────────────────────────────────────────────
def build_transaction_intelligence():

    print("Building Transaction Intelligence...")

    transactions = spark.read.table(
        "bib_catalog.silver.transactions"
    )

    customers = spark.read.table(
        "bib_catalog.silver.customers"
    ).filter(
        F.col("scd_is_current") == True
    ).select(
        "customer_id", "segment",
        "income_band", "city", "state"
    )

    txn_enriched = transactions.join(
        customers,
        on="customer_id",
        how="left"
    )

    # ── Daily aggregations ─────────────────
    daily_agg = (
        txn_enriched
        .withColumn(
            "txn_date",
            F.to_date(F.col("txn_datetime"))
        )
        .groupBy(
            "txn_date", "txn_type",
            "channel", "merchant_category",
            "status", "segment"
        )
        .agg(
            F.count("txn_id")
             .alias("txn_count"),
            F.sum("amount")
             .alias("total_amount"),
            F.avg("amount")
             .alias("avg_amount"),
            F.max("amount")
             .alias("max_amount"),
            F.min("amount")
             .alias("min_amount"),
            F.countDistinct("customer_id")
             .alias("unique_customers")
        )
        .withColumn(
            "txn_year",
            F.year(F.col("txn_date"))
        )
        .withColumn(
            "txn_month",
            F.month(F.col("txn_date"))
        )
        .withColumn(
            "txn_week",
            F.weekofyear(F.col("txn_date"))
        )
    )

    return write_gold(
        daily_agg,
        "bib_catalog.gold.transaction_intelligence",
        z_order_col="txn_date"
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 6: Gold Table 3 — Campaign Performance
# Conversion funnel + ROI metrics
# ─────────────────────────────────────────────
def build_campaign_performance():

    print("Building Campaign Performance...")

    campaign_interactions = spark.read.table(
        "bib_catalog.silver.campaign_interactions"
    )

    customers = spark.read.table(
        "bib_catalog.silver.customers"
    ).filter(
        F.col("scd_is_current") == True
    ).select(
        "customer_id", "segment",
        "income_band", "city", "state"
    )

    # ── REPLACE WITH ───────────────────────
    products = read_products_from_raw().select(
        "product_id", "product_name",
        "category", "sub_category"
    )

    enriched = (
        campaign_interactions
        .join(customers, on="customer_id", how="left")
        .join(products,  on="product_id",  how="left")
    )

    campaign_perf = (
        enriched
        .withColumn(
            "interaction_date",
            F.to_date(F.col("interaction_datetime"))
        )
        .groupBy(
            "campaign_id", "campaign_name",
            "product_id", "product_name",
            "category", "interaction_date",
            "segment"
        )
        .agg(
            F.count("interaction_id")
             .alias("total_interactions"),
            F.sum(
                F.when(
                    F.col("action_type") == "Sent", 1
                ).otherwise(0)
            ).alias("sent_count"),
            F.sum(
                F.when(
                    F.col("action_type") == "Opened", 1
                ).otherwise(0)
            ).alias("opened_count"),
            F.sum(
                F.when(
                    F.col("action_type") == "Clicked", 1
                ).otherwise(0)
            ).alias("clicked_count"),
            F.sum(
                F.when(
                    F.col("action_type") == "Converted", 1
                ).otherwise(0)
            ).alias("converted_count"),
            F.sum("converted")
             .alias("total_conversions"),
            F.countDistinct("customer_id")
             .alias("unique_customers")
        )
        # ── Funnel rate calculations ───────
        .withColumn(
            "open_rate",
            F.round(
                F.col("opened_count") /
                F.when(
                    F.col("sent_count") == 0, None
                ).otherwise(F.col("sent_count")) * 100,
                2
            )
        )
        .withColumn(
            "click_rate",
            F.round(
                F.col("clicked_count") /
                F.when(
                    F.col("opened_count") == 0, None
                ).otherwise(F.col("opened_count")) * 100,
                2
            )
        )
        .withColumn(
            "conversion_rate",
            F.round(
                F.col("converted_count") /
                F.when(
                    F.col("sent_count") == 0, None
                ).otherwise(F.col("sent_count")) * 100,
                2
            )
        )
    )

    return write_gold(
        campaign_perf,
        "bib_catalog.gold.campaign_performance",
        z_order_col="campaign_id"
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 7: Gold Table 4 — Branch Performance
# Zone → Region → Branch rollup
# ─────────────────────────────────────────────
def build_branch_performance():

    print("Building Branch Performance...")

    customers = spark.read.table(
        "bib_catalog.silver.customers"
    ).filter(
        F.col("scd_is_current") == True
    )

    transactions = spark.read.table(
        "bib_catalog.silver.transactions"
    )

    product_holdings = spark.read.table(
        "bib_catalog.silver.product_holdings"
    ).filter(
        F.col("scd_is_current") == True
    )

    branches = spark.read.table(
        "bib_catalog.silver.branches"
    )

    bank_managers = spark.read.table(
        "bib_catalog.silver.bank_managers"
    )

    # ── Customer count per branch ──────────
    customer_branch = (
        customers
        .groupBy("branch_id")
        .agg(
            F.count("customer_id")
             .alias("total_customers"),
            F.countDistinct("segment")
             .alias("unique_segments"),
            F.sum(
                F.when(
                    F.col("kyc_status") == "Verified", 1
                ).otherwise(0)
            ).alias("verified_customers")
        )
    )

    # ── Transaction value per branch ───────
    txn_branch = (
        transactions
        .join(
            customers.select(
                "customer_id", "branch_id"
            ),
            on="customer_id",
            how="left"
        )
        .groupBy("branch_id")
        .agg(
            F.count("txn_id")
             .alias("total_transactions"),
            F.sum("amount")
             .alias("total_txn_value"),
            F.avg("amount")
             .alias("avg_txn_value")
        )
    )

    # ── Portfolio value per branch ─────────
    holdings_branch = (
        product_holdings
        .join(
            customers.select(
                "customer_id", "branch_id"
            ),
            on="customer_id",
            how="left"
        )
        .groupBy("branch_id")
        .agg(
            F.sum("value_amount")
             .alias("total_portfolio_value"),
            F.count("holding_id")
             .alias("total_holdings")
        )
    )

    # ── Manager count per branch ───────────
    manager_branch = (
        bank_managers
        .groupBy("branch_id")
        .agg(
            F.count("manager_id")
             .alias("manager_count")
        )
    )

    # ── Join all ───────────────────────────
    branch_perf = (
        branches
        .join(customer_branch,
              on="branch_id", how="left")
        .join(txn_branch,
              on="branch_id", how="left")
        .join(holdings_branch,
              on="branch_id", how="left")
        .join(manager_branch,
              on="branch_id", how="left")
        .select(
            "branch_id", "branch_name",
            "branch_code", "branch_type",
            "city", "state", "zone", "region",
            F.coalesce(
                F.col("total_customers"),
                F.lit(0)
            ).alias("total_customers"),
            F.coalesce(
                F.col("verified_customers"),
                F.lit(0)
            ).alias("verified_customers"),
            F.coalesce(
                F.col("total_transactions"),
                F.lit(0)
            ).alias("total_transactions"),
            F.coalesce(
                F.col("total_txn_value"),
                F.lit(0.0)
            ).alias("total_txn_value"),
            F.coalesce(
                F.col("avg_txn_value"),
                F.lit(0.0)
            ).alias("avg_txn_value"),
            F.coalesce(
                F.col("total_portfolio_value"),
                F.lit(0.0)
            ).alias("total_portfolio_value"),
            F.coalesce(
                F.col("total_holdings"),
                F.lit(0)
            ).alias("total_holdings"),
            F.coalesce(
                F.col("manager_count"),
                F.lit(0)
            ).alias("manager_count")
        )
    )

    return write_gold(
        branch_perf,
        "bib_catalog.gold.branch_performance",
        z_order_col="branch_id"
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 8: Gold Table 5 — Product Sales Summary
# Product penetration + holding metrics
# ─────────────────────────────────────────────
def build_product_sales_summary():

    print("Building Product Sales Summary...")

    products = read_products_from_raw()

    product_holdings = spark.read.table(
        "bib_catalog.silver.product_holdings"
    ).filter(
        F.col("scd_is_current") == True
    )

    customers = spark.read.table(
        "bib_catalog.silver.customers"
    ).filter(
        F.col("scd_is_current") == True
    ).select(
        "customer_id", "segment",
        "income_band", "city", "state"
    )

    campaign_interactions = spark.read.table(
        "bib_catalog.silver.campaign_interactions"
    )

    # ── Campaign conversions per product ───
    campaign_conv = (
        campaign_interactions
        .groupBy("product_id")
        .agg(
            F.sum("converted")
             .alias("campaign_conversions"),
            F.count("interaction_id")
             .alias("campaign_interactions")
        )
    )

    product_summary = (
        product_holdings
        .join(customers,
              on="customer_id", how="left")
        .join(products,
              on="product_id",  how="left")
        .groupBy(
            "product_id", "product_name",
            "category", "sub_category",
            "segment", "income_band"
        )
        .agg(
            F.count("holding_id")
             .alias("total_holdings"),
            F.sum(
                F.when(
                    F.col("status") == "Active", 1
                ).otherwise(0)
            ).alias("active_holdings"),
            F.sum(
                F.when(
                    F.col("status") == "Closed", 1
                ).otherwise(0)
            ).alias("closed_holdings"),
            F.sum("value_amount")
             .alias("total_value"),
            F.avg("value_amount")
             .alias("avg_value"),
            F.countDistinct("customer_id")
             .alias("unique_customers")
        )
        .join(
            campaign_conv,
            on="product_id",
            how="left"
        )
        .withColumn(
            "active_rate",
            F.round(
                F.col("active_holdings") /
                F.when(
                    F.col("total_holdings") == 0, None
                ).otherwise(
                    F.col("total_holdings")
                ) * 100,
                2
            )
        )
    )

    return write_gold(
        product_summary,
        "bib_catalog.gold.product_sales_summary",
        z_order_col="product_id"
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 9: Gold Table 6 — User Access Mapping
# For RBAC in Power BI and Streamlit
# ─────────────────────────────────────────────
def build_user_access_mapping():

    print("Building User Access Mapping...")

    bank_managers = spark.read.table(
        "bib_catalog.silver.bank_managers"
    )

    branches = spark.read.table(
        "bib_catalog.silver.branches"
    ).select(
        "branch_id", "zone", "region"
    )

    user_mapping = (
        bank_managers
        .join(branches,
              on="branch_id", how="left")
        .select(
            F.col("email")
             .alias("user_email"),
            F.col("role"),
            F.col("manager_id"),
            bank_managers.branch_id,
            F.coalesce(
                branches.zone,
                bank_managers.zone
            ).alias("zone"),
            F.coalesce(
                branches.region,
                bank_managers.region
            ).alias("region"),
            F.col("is_active")
        )
    )

    return write_gold(
        user_mapping,
        "bib_catalog.gold.user_access_mapping",
        z_order_col="user_email"
    )

In [0]:
def build_product_regional_performance():

    print("Building Product Regional Performance...")

    product_holdings = spark.read.table(
        "bib_catalog.silver.product_holdings"
    ).filter(F.col("scd_is_current") == True)

    customers = spark.read.table(
        "bib_catalog.silver.customers"
    ).filter(
        F.col("scd_is_current") == True
    ).select(
        "customer_id",
        "branch_id",
        "segment",
        "income_band",
        "city",
        "state"
    )

    branches = spark.read.table(
        "bib_catalog.silver.branches"
    ).select(
        "branch_id",
        "branch_name",
        "branch_code",
        "branch_type",
        "city",
        "state",
        "zone",
        "region"
    )

    products = read_products_from_raw().select(
        "product_id",
        "product_name",
        "category",
        "sub_category",
        "min_income_required",
        "is_active"
    )

    campaign_interactions = spark.read.table(
        "bib_catalog.silver.campaign_interactions"
    )

    # Campaign conversions per product + branch
    campaign_branch = (
        campaign_interactions
        .join(
            customers.select(
                "customer_id",
                "branch_id"
            ),
            on="customer_id",
            how="left"
        )
        .groupBy(
            "product_id",
            "branch_id"
        )
        .agg(
            F.sum("converted")
             .alias("campaign_conversions"),

            F.count("interaction_id")
             .alias("campaign_interactions"),

            F.sum(
                F.when(
                    F.col("action_type") == "Sent",
                    1
                ).otherwise(0)
            ).alias("campaign_sent"),

            F.sum(
                F.when(
                    F.col("action_type") == "Opened",
                    1
                ).otherwise(0)
            ).alias("campaign_opened")
        )
    )

    # Holdings enriched with customer + branch
    holdings_enriched = (
        product_holdings
        .join(
            customers.select(
                "customer_id",
                "branch_id",
                "segment",
                "income_band",
                F.col("city").alias("customer_city"),
                F.col("state").alias("customer_state")
            ),
            on="customer_id",
            how="left"
        )
        .join(
            branches.select(
                "branch_id",
                "branch_name",
                "branch_code",
                "branch_type",
                "zone",
                "region",
                F.col("city").alias("branch_city"),
                F.col("state").alias("branch_state")
            ),
            on="branch_id",
            how="left"
        )
        .join(
            products,
            on="product_id",
            how="left"
        )
    )

    # Main aggregation
    product_regional = (
        holdings_enriched
        .groupBy(
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "is_active",
            "branch_id",
            "branch_name",
            "branch_type",
            "zone",
            "region",
            "branch_city",
            "branch_state",
            "segment",
            "income_band"
        )
        .agg(
            F.count("holding_id")
             .alias("total_holdings"),

            F.sum(
                F.when(
                    F.col("status") == "Active",
                    1
                ).otherwise(0)
            ).alias("active_holdings"),

            F.sum(
                F.when(
                    F.col("status") == "Closed",
                    1
                ).otherwise(0)
            ).alias("closed_holdings"),

            F.sum("value_amount")
             .alias("total_value"),

            F.avg("value_amount")
             .alias("avg_value"),

            F.max("value_amount")
             .alias("max_value"),

            F.countDistinct("customer_id")
             .alias("unique_customers")
        )
        .join(
            campaign_branch,
            on=["product_id", "branch_id"],
            how="left"
        )
        .withColumnRenamed(
            "branch_city",
            "city"
        )
        .withColumnRenamed(
            "branch_state",
            "state"
        )
        .withColumn(
            "active_rate",
            F.round(
                (
                    F.col("active_holdings")
                    / F.when(
                        F.col("total_holdings") == 0,
                        None
                    ).otherwise(
                        F.col("total_holdings")
                    )
                ) * 100,
                2
            )
        )
        .withColumn(
            "campaign_conversion_rate",
            F.round(
                (
                    F.col("campaign_conversions")
                    / F.when(
                        F.col("campaign_sent") == 0,
                        None
                    ).otherwise(
                        F.col("campaign_sent")
                    )
                ) * 100,
                2
            )
        )
        .withColumn(
            "avg_value_per_customer",
            F.round(
                F.col("total_value")
                / F.when(
                    F.col("unique_customers") == 0,
                    None
                ).otherwise(
                    F.col("unique_customers")
                ),
                2
            )
        )
        .fillna(
            0,
            subset=[
                "campaign_conversions",
                "campaign_interactions",
                "campaign_sent",
                "campaign_opened",
                "campaign_conversion_rate"
            ]
        )
    )

    return write_gold(
        product_regional,
        "bib_catalog.gold.product_regional_performance",
        z_order_col="product_id"
    )

In [0]:
# ─────────────────────────────────────────────
# Cell 10: Execute All Gold Tables + AI Insights
# ─────────────────────────────────────────────
gold_tables = [
    {
        "name":          "customer_360",
        "function":      build_customer_360,
        "insight_fn":    generate_customer_360_insight,
        "insight_type":  "Customer Health Summary"
    },
    {
        "name":          "transaction_intelligence",
        "function":      build_transaction_intelligence,
        "insight_fn":    generate_transaction_insight,
        "insight_type":  "Transaction Pattern Analysis"
    },
    {
        "name":          "campaign_performance",
        "function":      build_campaign_performance,
        "insight_fn":    generate_campaign_insight,
        "insight_type":  "Campaign Effectiveness Analysis"
    },
    {
        "name":          "branch_performance",
        "function":      build_branch_performance,
        "insight_fn":    generate_branch_insight,
        "insight_type":  "Branch Operations Summary"
    },
    {
        "name":          "product_sales_summary",
        "function":      build_product_sales_summary,
        "insight_fn":    None,
        "insight_type":  None
    },
    {
        "name":          "user_access_mapping",
        "function":      build_user_access_mapping,
        "insight_fn":    None,
        "insight_type":  None
    },
    {
        "name":         "product_regional_performance",
        "function":     build_product_regional_performance,
        "insight_fn":   generate_product_insight,
        "insight_type": "Product Performance Analysis"
    }
]

total_success = 0
total_failed  = 0

for gold in gold_tables:
    try:
        print(f"\n{'='*45}")
        print(f"Building : {gold['name']}")
        print(f"{'='*45}")

        records = gold["function"]()

        log_pipeline_event(
            notebook_name=notebook_name,
            pipeline_name=pipeline_name,
            source_name=gold["name"],
            layer=layer,
            status="SUCCESS",
            records_processed=records,
            message=(
                f"Gold table built: "
                f"bib_catalog.gold.{gold['name']}"
            ),
            environment=environment
        )
        print(
            f"SUCCESS  : {gold['name']} "
            f"| {records} records"
        )

        # ── Generate AI Insight ────────────
        if gold["insight_fn"]:
            try:
                print(
                    f"Generating AI insight for "
                    f"{gold['name']}..."
                )
                gold_df = spark.read.table(
                    f"bib_catalog.gold.{gold['name']}"
                )

                # Special case — anomaly detection
                # on transaction intelligence
                if gold["name"] == "transaction_intelligence":
                    anomaly_insight = (
                        generate_anomaly_detection_insight(
                            gold_df
                        )
                    )
                    write_ai_insight(
                        table_name=gold["name"],
                        insight_text=anomaly_insight,
                        insight_type="Anomaly Detection",
                        source_table=(
                            f"bib_catalog.gold."
                            f"{gold['name']}"
                        )
                    )
                    print(
                        f"Anomaly Insight:\n"
                        f"{anomaly_insight}\n"
                    )

                # Standard insight
                insight = gold["insight_fn"](gold_df)

                write_ai_insight(
                    table_name=gold["name"],
                    insight_text=insight,
                    insight_type=gold["insight_type"],
                    source_table=(
                        f"bib_catalog.gold."
                        f"{gold['name']}"
                    )
                )
                print(f"AI Insight:\n{insight}\n")

            except Exception as ai_err:
                # AI failure never blocks pipeline
                print(
                    f"AI insight failed for "
                    f"{gold['name']}: {str(ai_err)}"
                )

        total_success += 1

    except Exception as e:
        log_pipeline_event(
            notebook_name=notebook_name,
            pipeline_name=pipeline_name,
            source_name=gold["name"],
            layer=layer,
            status="FAILED",
            records_processed=0,
            message=(
                f"Gold build failed for "
                f"{gold['name']}: {str(e)}"
            ),
            environment=environment
        )
        print(
            f"FAILED   : {gold['name']} "
            f"| {str(e)}"
        )
        total_failed += 1

print(f"\n{'='*45}")
print(f"Gold Layer Complete")
print(f"Success  : {total_success}")
print(f"Failed   : {total_failed}")
print(f"{'='*45}")

if total_failed > 0:
    raise Exception(
        f"{total_failed} gold table(s) failed."
    )

In [0]:
%sql
SELECT
    insight_type,
    table_name,
    insight_text,
    generated_at
FROM bib_catalog.gold.ai_insights
ORDER BY generated_at DESC